In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os, subprocess, sys
from google.colab import files

# Проверяем, что GPU активен
!nvidia-smi

# ------------------------------------------------------------
# 1. Функция генерации матриц
def generate_matrices(n):
    os.makedirs("data", exist_ok=True)
    A = np.random.uniform(1, 10, (n, n)).astype(np.float32)
    B = np.random.uniform(1, 10, (n, n)).astype(np.float32)
    np.savetxt("data/A.txt", A, fmt="%.6f")
    np.savetxt("data/B.txt", B, fmt="%.6f")
    print(f"[GEN] Сгенерированы матрицы {n}x{n}")

# ------------------------------------------------------------
# 2. Шаблон CUDA-программы (подставим BLOCK_SIZE позже)
CUDA_TEMPLATE = r"""
#include <iostream>
#include <fstream>
#include <vector>
#include <cstdlib>
#include <iomanip>
#include <cuda_runtime.h>

#define BLOCK_SIZE %d

std::vector<float> read_matrix(const std::string& fname, int n) {
    std::ifstream f(fname);
    std::vector<float> m(n*n);
    for (int i = 0; i < n*n; ++i) f >> m[i];
    return m;
}

void write_matrix(const std::string& fname, const float* m, int n) {
    std::ofstream f(fname);
    f << std::fixed << std::setprecision(15);
    for (int i = 0; i < n; ++i) {
        for (int j = 0; j < n; ++j) {
            f << m[i*n + j] << (j < n-1 ? " " : "");
        }
        f << "\n";
    }
}

__global__ void mmul(const float* A, const float* B, float* C, int n) {
    __shared__ float As[BLOCK_SIZE][BLOCK_SIZE];
    __shared__ float Bs[BLOCK_SIZE][BLOCK_SIZE];
    int bx = blockIdx.x, by = blockIdx.y;
    int tx = threadIdx.x, ty = threadIdx.y;
    int row = by*BLOCK_SIZE + ty;
    int col = bx*BLOCK_SIZE + tx;
    float sum = 0.0f;
    for (int t = 0; t < (n+BLOCK_SIZE-1)/BLOCK_SIZE; ++t) {
        int ar = row, ac = t*BLOCK_SIZE + tx;
        As[ty][tx] = (ar < n && ac < n) ? A[ar*n + ac] : 0.0f;
        int br = t*BLOCK_SIZE + ty, bc = col;
        Bs[ty][tx] = (br < n && bc < n) ? B[br*n + bc] : 0.0f;
        __syncthreads();
        for (int k = 0; k < BLOCK_SIZE; ++k) sum += As[ty][k] * Bs[k][tx];
        __syncthreads();
    }
    if (row < n && col < n) C[row*n + col] = sum;
}

int main(int argc, char* argv[]) {
    int n = std::atoi(argv[1]);
    auto hA = read_matrix("data/A.txt", n);
    auto hB = read_matrix("data/B.txt", n);
    std::vector<float> hC(n*n, 0.0f);

    float *dA, *dB, *dC;
    cudaMalloc(&dA, n*n*sizeof(float));
    cudaMalloc(&dB, n*n*sizeof(float));
    cudaMalloc(&dC, n*n*sizeof(float));
    cudaMemcpy(dA, hA.data(), n*n*sizeof(float), cudaMemcpyHostToDevice);
    cudaMemcpy(dB, hB.data(), n*n*sizeof(float), cudaMemcpyHostToDevice);

    dim3 threads(BLOCK_SIZE, BLOCK_SIZE);
    dim3 blocks((n+BLOCK_SIZE-1)/BLOCK_SIZE, (n+BLOCK_SIZE-1)/BLOCK_SIZE);

    cudaEvent_t start, stop;
    cudaEventCreate(&start); cudaEventCreate(&stop);
    cudaEventRecord(start);
    mmul<<<blocks, threads>>>(dA, dB, dC, n);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    float ms = 0;
    cudaEventElapsedTime(&ms, start, stop);

    cudaMemcpy(hC.data(), dC, n*n*sizeof(float), cudaMemcpyDeviceToHost);
    write_matrix("data/C.txt", hC.data(), n);

    cudaFree(dA); cudaFree(dB); cudaFree(dC);
    cudaEventDestroy(start); cudaEventDestroy(stop);

    std::cout << "Размер матрицы: " << n << "x" << n << "\n";
    std::cout << "Время: " << ms/1000.0 << " секунд\n";
    std::cout << "Производительность: " << (2.0*n*n*n)/(ms/1000.0)/1e9 << " GFLOPS\n";
    return 0;
}
"""

# ------------------------------------------------------------
# 3. Функция компиляции и тестирования для одного BLOCK_SIZE
def run_for_block(block_size, sizes):
    # Записываем .cu файл с подставленным размером блока
    with open("matmul.cu", "w") as f:
        f.write(CUDA_TEMPLATE % block_size)
    # Компиляция (arch=sm_75 для T4; если используете другую GPU, измените)
    ret = subprocess.run(
        f"nvcc -DBLOCK_SIZE={block_size} -O2 -arch=sm_75 matmul.cu -o matmul",
        shell=True, capture_output=True, text=True
    )
    if ret.returncode != 0:
        print("Ошибка компиляции:", ret.stderr)
        return []

    results = []
    for N in sizes:
        generate_matrices(N)
        best_t = None
        # Три запуска, берём минимальное время
        for _ in range(3):
            res = subprocess.run(["./matmul", str(N)], capture_output=True, text=True)
            for line in res.stdout.splitlines():
                if line.startswith("Время:"):
                    t = float(line.split()[1])
                    if best_t is None or t < best_t:
                        best_t = t
        if best_t is None:
            print(f"[FAIL] Не получено время для N={N}, block={block_size}")
            continue
        # Верификация
        A = np.loadtxt("data/A.txt", dtype=np.float32).reshape(N, N)
        B = np.loadtxt("data/B.txt", dtype=np.float32).reshape(N, N)
        C = np.loadtxt("data/C.txt", dtype=np.float32).reshape(N, N)
        C_np = A @ B
        if not np.allclose(C, C_np, rtol=1e-4, atol=1e-5):
            print(f"[FAIL] Верификация не пройдена для N={N}, block={block_size}")
            continue
        gflops = (2.0 * N**3) / best_t / 1e9
        results.append((block_size, N, best_t, gflops))
        print(f"[OK] N={N:4d}, block={block_size:2d}, time={best_t:.6f} s, GFLOPS={gflops:.2f}")
    return results

# ------------------------------------------------------------
# 4. Параметры экспериментов (можно менять)
MATRIX_SIZES = [256, 512, 1024, 2048]   # для T4 хватит памяти
BLOCK_SIZES  = [16, 32]

all_data = []
for bs in BLOCK_SIZES:
    all_data.extend(run_for_block(bs, MATRIX_SIZES))

# ------------------------------------------------------------
# 5. Сохраняем результаты и строим график
df = pd.DataFrame(all_data, columns=["BlockSize", "N", "Time", "GFLOPS"])
df.to_csv("performance_results_cuda.csv", index=False)
print("\n=== ИТОГОВАЯ ТАБЛИЦА ===")
print(df.to_string(index=False))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
for bs in BLOCK_SIZES:
    sub = df[df["BlockSize"] == bs]
    ax1.plot(sub["N"], sub["Time"], 'o-', label=f"BLOCK_SIZE={bs}")
    ax2.plot(sub["N"], sub["GFLOPS"], 's--', label=f"BLOCK_SIZE={bs}")
ax1.set_xlabel("Размер матрицы N"), ax1.set_ylabel("Время (с)")
ax1.legend(), ax1.grid(True)
ax2.set_xlabel("Размер матрицы N"), ax2.set_ylabel("GFLOPS")
ax2.legend(), ax2.grid(True)
plt.tight_layout()
plt.savefig("performance_plot_cuda.png")
plt.show()

# ------------------------------------------------------------
# 6. Скачивание результатов на ваш компьютер
files.download("performance_results_cuda.csv")
files.download("performance_plot_cuda.png")